# Modelado Base (Baselines) y Orquestador de Validación Operativa

El objetivo de este cuaderno es establecer la línea base de rendimiento de algoritmos clásicos de Machine Learning (TF-IDF combinado con clasificadores estadísticos) ante el problema industrial Telco (56 clases).

Para prevenir la fuga de datos (*data leakage*), la experimentación se restringe exclusivamente al conjunto de entrenamiento. Se construye un evaluador personalizado que itera sobre las particiones pre-calculadas. Este orquestador no solo captura métricas académicas (Log-Loss, F1, Kappa), sino que simula las reglas de negocio imponiendo un "Cortafuegos de Pasividad" brutal: toda predicción que no alcance un umbral de confianza del **0.85** será clasificada como `MANUAL_REVIEW`. Esta intercepción revelará si los modelos clásicos están realmente calibrados para automatizar operaciones en un BPO sin destruir el SLA.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss, f1_score, cohen_kappa_score
import time
import warnings

# Bloqueamos warnings de Scikit-Learn por etiquetas faltantes en pliegues pequeños
warnings.filterwarnings('ignore')

# 1. Ingesta de datos (Conjunto de entrenamiento exclusivo)
path_train = "../data/gold/train_set_telco.parquet"
df_train = pd.read_parquet(path_train)

# Extracción de variables según la arquitectura de la Fase 1
X = df_train['full_text']
y = df_train['target_tripleta']
folds = df_train['fold_id']

# Inspección de clases únicas para dimensionar las matrices de Log-Loss
clases_unicas = np.sort(y.unique())
print(f"Dataset cargado -> Volumen: {len(X)} tickets | Clases: {len(clases_unicas)}")

# 2. Definición del orquestador de validación y simulación de reglas de negocio
def evaluar_modelo_bpo(pipeline_modelo, X, y, folds, umbral_confianza=0.85):
    """
    Evalúa un pipeline utilizando validación cruzada sobre particiones estáticas.
    Interviene las predicciones probabilísticas aplicando un umbral mínimo de confianza 
    para simular el enrutamiento a agentes humanos.
    """
    resultados = []
    
    # Iteración determinista sobre los 5 particionados
    for fold_actual in sorted(folds.unique()):
        
        # Máscaras booleanas para segregación de train y validation
        idx_train = folds != fold_actual
        idx_val = folds == fold_actual
        
        # Filtrado posicional en Pandas (evitando colisiones de .iloc con booleanos)
        X_train, y_train = X[idx_train], y[idx_train]
        X_val, y_val = X[idx_val], y[idx_val]
        
        # --- Medición de latencia de entrenamiento ---
        t_inicio = time.time()
        pipeline_modelo.fit(X_train, y_train)
        t_fit = time.time() - t_inicio
        
        # --- Medición de latencia de inferencia ---
        t_inicio_inf = time.time()
        y_proba = pipeline_modelo.predict_proba(X_val)
        t_inf = time.time() - t_inicio_inf
        latencia_ms = (t_inf / len(X_val)) * 1000
        
        clases_modelo = pipeline_modelo.classes_
        
        # --- Cálculo de métricas académicas (Previo a intercepción) ---
        indices_pred = np.argmax(y_proba, axis=1)
        y_pred_bruto = clases_modelo[indices_pred]
        
        ll = log_loss(y_val, y_proba, labels=clases_modelo)
        f1_mac = f1_score(y_val, y_pred_bruto, average='macro', zero_division=0)
        f1_wei = f1_score(y_val, y_pred_bruto, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_val, y_pred_bruto)
        
        # --- Simulación de reglas operativas (Umbral BPO) ---
        prob_maximas = np.max(y_proba, axis=1)
        
        # Sobrescritura de predicciones basándonos en la dispersión de la confianza
        y_pred_bpo = np.where(prob_maximas >= umbral_confianza, y_pred_bruto, 'MANUAL_REVIEW')
        
        # Conteo de observaciones que superan el umbral (volumen automatizable)
        idx_automatizados = y_pred_bpo != 'MANUAL_REVIEW'
        volumen_automatizado = np.sum(idx_automatizados)
        
        # Métrica BPO 1: Proporción del volumen total que el sistema asume
        tasa_automatizacion = volumen_automatizado / len(y_val)
        
        # Métrica BPO 2: Accuracy exclusivo sobre el volumen automatizado
        if volumen_automatizado > 0:
            # Uso de .values para garantizar alineación exacta de índices subyacentes
            precision_condicionada = np.mean(y_pred_bpo[idx_automatizados] == y_val.values[idx_automatizados])
        else:
            precision_condicionada = 0.0
            
        # Agrupación de la telemetría del pliegue actual
        resultados.append({
            'Fold': fold_actual,
            'Fit_Time_s': t_fit,
            'Latency_ms': latencia_ms,
            'Log_Loss': ll,
            'F1_Macro': f1_mac,
            'F1_Weighted': f1_wei,
            'Kappa': kappa,
            'Tasa_Automatizacion': tasa_automatizacion,
            'Precision_Condicionada': precision_condicionada
        })
        
    return pd.DataFrame(resultados)

Dataset cargado -> Volumen: 12322 tickets | Clases: 56


# Vectorización Léxica y Modelo A (Random Forest Calibrado)

La transformación del texto plano a tensores matemáticos se realiza mediante la frecuencia de término e inversa de documento (TF-IDF), restringiendo el hiperplano a las 15.000 dimensiones más relevantes.

Como modelo de control no lineal (Modelo A), se instancia un ensamble de árboles (`RandomForestClassifier`) con ponderación equilibrada de clases para compensar el desbalanceo originado por la *Power Law*. Se encapsula este estimador mediante la clase `CalibratedClassifierCV` para forzar que el algoritmo emita probabilidades reales, habilitando así el funcionamiento del umbral operativo (0.85).

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

print("Instanciando el Modelo A (Random Forest Calibrado)...")

# 1. Transformación del espacio léxico
# Se fija un techo dimensional estricto para evitar errores de MemoryError
vectorizador_tfidf = TfidfVectorizer(max_features=15000, stop_words='english')

# 2. Configuración del estimador base
# n_jobs=-1 maximiza el uso de CPU. class_weight='balanced' es crítico para el long tail.
rf_base = RandomForestClassifier(
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

# 3. Envoltura de calibración probabilística
# method='sigmoid' (Platt) previene el sobreajuste que causaría el método isótono en clases pequeñas.
# cv=2 asegura la viabilidad técnica del particionado sobre clases con bajo soporte.
rf_calibrado = CalibratedClassifierCV(
    estimator=rf_base, 
    method='sigmoid', 
    cv=2
)

# 4. Construcción del flujo secuencial
pipeline_rf = Pipeline([
    ('tfidf', vectorizador_tfidf),
    ('clf', rf_calibrado)
])

# 5. Ejecución del evaluador determinista
print("Iniciando validación cruzada. El cálculo de los tensores y la calibración tomará varios minutos...")
df_resultados_rf = evaluar_modelo_bpo(pipeline_rf, X, y, folds, umbral_confianza=0.85)
df_resultados_rf.to_csv("../metrics/resultados_kfold_nativo_telco.csv", index=False)

print("\n--- Telemetría Consolidada: Modelo A (Random Forest) ---")
# Cálculo de la media sobre los 5 folds para reportar el rendimiento estático
metricas_promedio_rf = df_resultados_rf.mean().drop('Fold')
print(metricas_promedio_rf.round(4))

Instanciando el Modelo A (Random Forest Calibrado)...
Iniciando validación cruzada. El cálculo de los tensores y la calibración tomará varios minutos...

--- Telemetría Consolidada: Modelo A (Random Forest) ---
Fit_Time_s                3.1149
Latency_ms                0.1100
Log_Loss                  1.8811
F1_Macro                  0.5842
F1_Weighted               0.5424
Kappa                     0.5099
Tasa_Automatizacion       0.0000
Precision_Condicionada    0.0000
dtype: float64


# Modelo B (Regresión Logística Multinomial)

Ante el previsible sufrimiento del Random Forest en espacios léxicos hiper-dispersos, se implementa un modelo de control lineal (`LogisticRegression`). Al optimizar estrictamente la pérdida logarítmica (Log-Loss) durante su descenso de gradiente iterativo, sus proyecciones probabilísticas están naturalmente ancladas, buscando comprobar si una separación lineal pura genera distribuciones que logren superar la barrera del 0.85 de confianza.

In [3]:
from sklearn.linear_model import LogisticRegression

print("Instanciando el Modelo B (Regresión Logística Multinomial)...")

# 1. Configuración del estimador de control lineal
# max_iter=1000 y solver='saga' garantizan convergencia sin saturar la RAM local en alta dimensión
lr_base = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000, 
    solver='saga',
    random_state=42,
    n_jobs=-1
)

# 2. Construcción del flujo secuencial
# Reutilizamos la misma parametrización del vectorizador para blindar la comparativa (Bake-Off)
pipeline_lr = Pipeline([
    ('tfidf', vectorizador_tfidf),
    ('clf', lr_base)
])

# 3. Ejecución del orquestador BPO determinista
print("Iniciando validación cruzada. El descenso de gradiente estocástico (SAGA) tomará unos minutos...")
df_resultados_lr = evaluar_modelo_bpo(pipeline_lr, X, y, folds, umbral_confianza=0.85)

print("\n--- Telemetría Consolidada: Modelo B (Regresión Logística) ---")
# Agrupación y extracción del rendimiento estático promedio
metricas_promedio_lr = df_resultados_lr.mean().drop('Fold')
print(metricas_promedio_lr.round(4))

Instanciando el Modelo B (Regresión Logística Multinomial)...
Iniciando validación cruzada. El descenso de gradiente estocástico (SAGA) tomará unos minutos...

--- Telemetría Consolidada: Modelo B (Regresión Logística) ---
Fit_Time_s                75.5848
Latency_ms                 0.0417
Log_Loss                   2.6340
F1_Macro                   0.3115
F1_Weighted                0.2971
Kappa                      0.2737
Tasa_Automatizacion        0.0017
Precision_Condicionada     0.9600
dtype: float64


# Síntesis Operativa y El Colapso Probabilístico del ML Clásico

El análisis matricial de la telemetría arroja un diagnóstico demoledor que justifica de facto el salto al Deep Learning:

1. **La Farsa del F1-Macro:** El Modelo A (Random Forest) logra un aparente rendimiento estadístico sólido (F1 ~0.58). Un perfil junior daría este modelo por válido. Sin embargo, este F1 es una simple ilusión de memorización léxica (TF-IDF actuando como diccionario in-domain).
2. **El Colapso del Cortafuegos (0.00% Automatización):** Al aplicarle el SLA de negocio (85% de confianza mínima), la Tasa de Automatización del Random Forest se estrella absolutamente al **0.00%**. El algoritmo "acierta" la clase, pero jamás está seguro de lo que hace. Su Log-Loss (1.88) evidencia una calibración mediocre (Hard Voting). 
3. **El Cuello de Botella Representacional:** El modelo clásico es un cobarde operativo. Es incapaz de asumir un solo ticket automáticamente bajo normas de QA estrictas. Esta incapacidad de gestionar la incertidumbre nos obliga estructuralmente a abandonar el TF-IDF e iterar hacia espacios semánticos profundos.

In [4]:
import pandas as pd

print("Consolidando métricas de validación cruzada para el reporte BPO...")

# Ensamblaje matricial de los rendimientos estáticos promedio
leaderboard_baselines = pd.DataFrame({
    'Modelo_A (RF_Calibrado)': metricas_promedio_rf,
    'Modelo_B (Reg_Logistica)': metricas_promedio_lr
}).T

# Reordenación de vectores para priorizar la lectura del impacto operativo
columnas_ordenadas = [
    'Tasa_Automatizacion', 
    'Precision_Condicionada', 
    'F1_Macro', 
    'F1_Weighted', 
    'Log_Loss', 
    'Kappa', 
    'Fit_Time_s', 
    'Latency_ms'
]

leaderboard_baselines = leaderboard_baselines[columnas_ordenadas]

print("\n--- LEADERBOARD DEFINITIVO: BASELINES LÉXICOS ---")
display(leaderboard_baselines.round(4))

Consolidando métricas de validación cruzada para el reporte BPO...

--- LEADERBOARD DEFINITIVO: BASELINES LÉXICOS ---


,Tasa_Automatizacion,Precision_Condicionada,F1_Macro,F1_Weighted,Log_Loss,Kappa,Fit_Time_s,Latency_ms
Modelo_A (RF_Calibrado),0.0000,0.00,0.5842,0.5424,1.8811,0.5099,3.1149,0.1100
Modelo_B (Reg_Logistica),0.0017,0.96,0.3115,0.2971,2.6340,0.2737,75.5848,0.0417
